In [1]:
import os

print("Files inside data folder:")
print(os.listdir("data"))

Files inside data folder:
['set-a', 'set-a.tar.gz']


In [2]:
patient_files = os.listdir("data/set-a")

print("Number of patient files:", len(patient_files))
print("First 5 files:", patient_files[:5])

Number of patient files: 1
First 5 files: ['set-a']


In [3]:
import os

print(os.listdir("data/set-a"))

['set-a']


In [4]:
print(os.listdir("data/set-a/set-a"))

['132539.txt', '132540.txt', '132541.txt', '132543.txt', '132545.txt', '132547.txt', '132548.txt', '132551.txt', '132554.txt', '132555.txt', '132556.txt', '132567.txt', '132568.txt', '132570.txt', '132573.txt', '132575.txt', '132577.txt', '132582.txt', '132584.txt', '132585.txt', '132588.txt', '132590.txt', '132591.txt', '132592.txt', '132595.txt', '132597.txt', '132598.txt', '132599.txt', '132601.txt', '132602.txt', '132605.txt', '132610.txt', '132612.txt', '132614.txt', '132615.txt', '132617.txt', '132618.txt', '132622.txt', '132623.txt', '132632.txt', '132634.txt', '132635.txt', '132636.txt', '132637.txt', '132639.txt', '132642.txt', '132644.txt', '132647.txt', '132648.txt', '132650.txt', '132653.txt', '132658.txt', '132659.txt', '132662.txt', '132663.txt', '132664.txt', '132666.txt', '132669.txt', '132671.txt', '132680.txt', '132682.txt', '132683.txt', '132685.txt', '132686.txt', '132688.txt', '132694.txt', '132695.txt', '132698.txt', '132700.txt', '132703.txt', '132704.txt', '1327

In [5]:
patient_files = os.listdir("data/set-a/set-a")

print("Number of patient files:", len(patient_files))
print("First 5 files:", patient_files[:5])

Number of patient files: 4000
First 5 files: ['132539.txt', '132540.txt', '132541.txt', '132543.txt', '132545.txt']


In [6]:
import pandas as pd
import os

# Correct path
patient_path = "data/set-a/set-a"

# Get patient files
patient_files = os.listdir(patient_path)

# Select first patient
first_patient = patient_files[0]

print("Loading:", first_patient)

# Full file path
file_path = os.path.join(patient_path, first_patient)

# Load file
patient_df = pd.read_csv(file_path)

# Show first 10 rows
patient_df.head(10)

Loading: 132539.txt


,Time,Parameter,Value
0,00:00,RecordID,132539.00
1,00:00,Age,54.00
2,00:00,Gender,0.00
3,00:00,Height,-1.00
4,00:00,ICUType,4.00
5,00:00,Weight,-1.00
6,00:07,GCS,15.00
7,00:07,HR,73.00
8,00:07,NIDiasABP,65.00
9,00:07,NIMAP,92.33


In [7]:
# Remove static variables first (Age, Gender, etc.)
static_vars = ["RecordID", "Age", "Gender", "Height", "ICUType", "Weight"]

dynamic_df = patient_df[~patient_df["Parameter"].isin(static_vars)]

# Pivot to wide format
wide_df = dynamic_df.pivot_table(
    index="Time",
    columns="Parameter",
    values="Value",
    aggfunc="mean"
)

wide_df.head()

Parameter,BUN,Creatinine,GCS,Glucose,HCO3,HCT,HR,K,Mg,NIDiasABP,NIMAP,NISysABP,Na,Platelets,RespRate,Temp,Urine,WBC
Time,,,,,,,,,,,,,,,,,,
00:07,NaN,NaN,15.0,NaN,NaN,NaN,73.0,NaN,NaN,65.0,92.33,147.0,NaN,NaN,19.0,35.1,900.0,NaN
00:37,NaN,NaN,NaN,NaN,NaN,NaN,77.0,NaN,NaN,58.0,91.00,157.0,NaN,NaN,19.0,35.6,60.0,NaN
01:37,NaN,NaN,NaN,NaN,NaN,NaN,60.0,NaN,NaN,62.0,87.00,137.0,NaN,NaN,18.0,NaN,30.0,NaN
02:37,NaN,NaN,NaN,NaN,NaN,NaN,62.0,NaN,NaN,52.0,75.67,123.0,NaN,NaN,19.0,NaN,170.0,NaN
03:08,NaN,NaN,NaN,NaN,NaN,33.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Reset index so Time becomes a column
wide_df = wide_df.reset_index()

# Convert Time (HH:MM) into total minutes
def time_to_minutes(t):
    h, m = t.split(":")
    return int(h) * 60 + int(m)

wide_df["Time_minutes"] = wide_df["Time"].apply(time_to_minutes)

# Sort by time
wide_df = wide_df.sort_values("Time_minutes")

wide_df.head()

Parameter,Time,BUN,Creatinine,GCS,Glucose,HCO3,HCT,HR,K,Mg,NIDiasABP,NIMAP,NISysABP,Na,Platelets,RespRate,Temp,Urine,WBC,Time_minutes
0,00:07,NaN,NaN,15.0,NaN,NaN,NaN,73.0,NaN,NaN,65.0,92.33,147.0,NaN,NaN,19.0,35.1,900.0,NaN,7
1,00:37,NaN,NaN,NaN,NaN,NaN,NaN,77.0,NaN,NaN,58.0,91.00,157.0,NaN,NaN,19.0,35.6,60.0,NaN,37
2,01:37,NaN,NaN,NaN,NaN,NaN,NaN,60.0,NaN,NaN,62.0,87.00,137.0,NaN,NaN,18.0,NaN,30.0,NaN,97
3,02:37,NaN,NaN,NaN,NaN,NaN,NaN,62.0,NaN,NaN,52.0,75.67,123.0,NaN,NaN,19.0,NaN,170.0,NaN,157
4,03:08,NaN,NaN,NaN,NaN,NaN,33.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,188


In [9]:
# Check missing values
missing_counts = wide_df.isnull().sum()

print("Missing values per column:")
print(missing_counts[missing_counts > 0].sort_values(ascending=False).head(10))

Missing values per column:
Parameter
BUN           48
Creatinine    48
Glucose       48
HCO3          48
Mg            48
K             48
Na            48
WBC           48
Platelets     48
HCT           47
dtype: int64


In [11]:
# Forward fill
wide_df_filled = wide_df.ffill()

# Fill remaining NaN (at beginning) with 0
wide_df_filled = wide_df_filled.fillna(0)

wide_df_filled.head()

Parameter,Time,BUN,Creatinine,GCS,Glucose,HCO3,HCT,HR,K,Mg,NIDiasABP,NIMAP,NISysABP,Na,Platelets,RespRate,Temp,Urine,WBC,Time_minutes
0,00:07,0.0,0.0,15.0,0.0,0.0,0.0,73.0,0.0,0.0,65.0,92.33,147.0,0.0,0.0,19.0,35.1,900.0,0.0,7
1,00:37,0.0,0.0,15.0,0.0,0.0,0.0,77.0,0.0,0.0,58.0,91.00,157.0,0.0,0.0,19.0,35.6,60.0,0.0,37
2,01:37,0.0,0.0,15.0,0.0,0.0,0.0,60.0,0.0,0.0,62.0,87.00,137.0,0.0,0.0,18.0,35.6,30.0,0.0,97
3,02:37,0.0,0.0,15.0,0.0,0.0,0.0,62.0,0.0,0.0,52.0,75.67,123.0,0.0,0.0,19.0,35.6,170.0,0.0,157
4,03:08,0.0,0.0,15.0,0.0,0.0,33.7,62.0,0.0,0.0,52.0,75.67,123.0,0.0,0.0,19.0,35.6,170.0,0.0,188


In [12]:
print("All available variables:")
print(wide_df_filled.columns.tolist())

All available variables:
['Time', 'BUN', 'Creatinine', 'GCS', 'Glucose', 'HCO3', 'HCT', 'HR', 'K', 'Mg', 'NIDiasABP', 'NIMAP', 'NISysABP', 'Na', 'Platelets', 'RespRate', 'Temp', 'Urine', 'WBC', 'Time_minutes']


In [13]:
# Drop string time column
model_df = wide_df_filled.drop(columns=["Time"])

# Set Time_minutes as index
model_df = model_df.set_index("Time_minutes")

model_df.head()

Parameter,BUN,Creatinine,GCS,Glucose,HCO3,HCT,HR,K,Mg,NIDiasABP,NIMAP,NISysABP,Na,Platelets,RespRate,Temp,Urine,WBC
Time_minutes,,,,,,,,,,,,,,,,,,
7,0.0,0.0,15.0,0.0,0.0,0.0,73.0,0.0,0.0,65.0,92.33,147.0,0.0,0.0,19.0,35.1,900.0,0.0
37,0.0,0.0,15.0,0.0,0.0,0.0,77.0,0.0,0.0,58.0,91.00,157.0,0.0,0.0,19.0,35.6,60.0,0.0
97,0.0,0.0,15.0,0.0,0.0,0.0,60.0,0.0,0.0,62.0,87.00,137.0,0.0,0.0,18.0,35.6,30.0,0.0
157,0.0,0.0,15.0,0.0,0.0,0.0,62.0,0.0,0.0,52.0,75.67,123.0,0.0,0.0,19.0,35.6,170.0,0.0
188,0.0,0.0,15.0,0.0,0.0,33.7,62.0,0.0,0.0,52.0,75.67,123.0,0.0,0.0,19.0,35.6,170.0,0.0


In [14]:
# Keep only first 48 hours (48 * 60 minutes = 2880)
model_df_48h = model_df[model_df.index <= 2880]

print("Time range:", model_df_48h.index.min(), "to", model_df_48h.index.max())
print("Shape:", model_df_48h.shape)

Time range: 7 to 2857
Shape: (50, 18)


In [15]:
# Convert minutes to hours
model_df_48h["Hour"] = model_df_48h.index // 60

# Group by hour and take mean
hourly_df = model_df_48h.groupby("Hour").mean()

hourly_df.head()

Parameter,BUN,Creatinine,GCS,Glucose,HCO3,HCT,HR,K,Mg,NIDiasABP,NIMAP,NISysABP,Na,Platelets,RespRate,Temp,Urine,WBC
Hour,,,,,,,,,,,,,,,,,,
0,0.0,0.0,15.0,0.0,0.0,0.0,75.0,0.0,0.0,61.5,91.665,152.0,0.0,0.0,19.0,35.35,480.0,0.0
1,0.0,0.0,15.0,0.0,0.0,0.0,60.0,0.0,0.0,62.0,87.000,137.0,0.0,0.0,18.0,35.60,30.0,0.0
2,0.0,0.0,15.0,0.0,0.0,0.0,62.0,0.0,0.0,52.0,75.670,123.0,0.0,0.0,19.0,35.60,170.0,0.0
3,0.0,0.0,15.0,0.0,0.0,33.7,71.0,0.0,0.0,52.0,74.170,118.5,0.0,0.0,19.5,36.70,115.0,0.0
4,0.0,0.0,15.0,0.0,0.0,33.7,74.0,0.0,0.0,52.0,72.670,114.0,0.0,0.0,20.0,37.80,60.0,0.0


In [16]:
print("Hourly shape:", hourly_df.shape)
print("Hours available:", hourly_df.index.min(), "to", hourly_df.index.max())

Hourly shape: (47, 18)
Hours available: 0 to 47


In [17]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit and transform
scaled_array = scaler.fit_transform(hourly_df)

# Convert back to DataFrame
scaled_df = pd.DataFrame(
    scaled_array,
    index=hourly_df.index,
    columns=hourly_df.columns
)

scaled_df.head()

Parameter,BUN,Creatinine,GCS,Glucose,HCO3,HCT,HR,K,Mg,NIDiasABP,NIMAP,NISysABP,Na,Platelets,RespRate,Temp,Urine,WBC
Hour,,,,,,,,,,,,,,,,,,
0,-1.833071,-2.032994,0.304997,-1.767268,-2.047697,-3.764004,0.340063,-2.043501,-1.984018,1.648739,2.597838,3.356107,-2.054736,-2.017022,0.446878,-3.233382,2.413626,-2.018067
1,-1.833071,-2.032994,0.304997,-1.767268,-2.047697,-3.764004,-1.442421,-2.043501,-1.984018,1.715830,2.026194,2.066352,-2.054736,-2.017022,0.131039,-2.860909,-1.097254,-2.018067
2,-1.833071,-2.032994,0.304997,-1.767268,-2.047697,-3.764004,-1.204757,-2.043501,-1.984018,0.374000,0.637828,0.862581,-2.054736,-2.017022,0.446878,-2.860909,-0.004980,-2.018067
3,-1.833071,-2.032994,0.304997,-1.767268,-2.047697,0.413270,-0.135267,-2.043501,-1.984018,0.374000,0.454020,0.475654,-2.054736,-2.017022,0.604797,-1.222028,-0.434088,-2.018067
4,-1.833071,-2.032994,0.304997,-1.767268,-2.047697,0.413270,0.221230,-2.043501,-1.984018,0.374000,0.270211,0.088728,-2.054736,-2.017022,0.762716,0.416853,-0.863195,-2.018067


In [18]:
import torch
import numpy as np

# Convert to numpy
data_array = scaled_df.values

# Add batch dimension
data_array = np.expand_dims(data_array, axis=0)

# Convert to tensor
tensor_data = torch.tensor(data_array, dtype=torch.float32)

print("Tensor shape:", tensor_data.shape)

Tensor shape: torch.Size([1, 47, 18])
